This analysis examines my stepcount and heartrate over my trip to the USA and Canada

The source of the data is Gadgetbridge and a Huawei Band 7

Gadgetbridge is an app that allows connection to a host of wearables without needing to connect to an external server. All data is stored locally, and can be exported as an SQLite dataset

this dataset has been manually loaded into databricks

In [0]:
# Get data from SQLite file exported from device and uploaded to databricks
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession

# timezone of travel was the New York timezone, set this so the timestamps automatically convert to the time I was experiencing
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

spark = SparkSession.builder.getOrCreate()

conn = sqlite3.connect("//Volumes/workspace/default/test/Gadgetbridge")

# List all tables in Gadgetbridge
tables = pd.read_sql_query(
"SELECT name FROM sqlite_master WHERE type='table';", conn
)

#HUAWEI_ACTIVITY_SAMPLE is the table that tracks all my data
# Every second entry has a stepcount of -1, difficult to determine why this is, can't find documentation on how GadgetBridge records data
pd_activity = pd.read_sql_query(f'''SELECT TIMESTAMP,
                                    OTHER_TIMESTAMP,
                                    STEPS,
                                    HEART_RATE,
                                    SOURCE
                                FROM HUAWEI_ACTIVITY_SAMPLE
                                WHERE STEPS > -1
                                ''', conn)
conn.close()






In [0]:
  # display(activity)

  # convert from Screaming Snake Case to snake case
  cols = pd_activity.columns

  cols = cols.str.lower()
  pd_activity.columns = cols
  # Transform timestamps into days

from pyspark.sql import functions as f
from pyspark.sql import types as t

activity = spark.createDataFrame(pd_activity)

activity = activity.withColumn('timestamp_date', f.to_timestamp(activity.timestamp.cast(dataType=t.TimestampType()),'V'))
activity = activity.withColumn('timestamp_date_other', f.to_timestamp(activity.other_timestamp.cast(dataType=t.TimestampType())))

display(activity)



In [0]:
# clean data for time period I care about
activity.createOrReplaceTempView('activity')
na_activity = spark.sql('''
                        select
                        date(timestamp_date) as date ,
                        sum(steps) as steps,
                        round(avg(case when heart_rate > 0 then heart_rate
                            else null end), 2) as heart_rate,
                            case when date(timestamp_date) < '2026-01-27' then 'NYC'
                                 when date(timestamp_date) < '2026-01-30' then 'Philadelphia'
                                 when date(timestamp_date) < '2026-02-02' then 'DC'
                                 when date(timestamp_date) < '2026-02-05' then 'Toronto'
                                 when date(timestamp_date) < '2026-02-08' then 'Ottawa'
                                 when date(timestamp_date) < '2026-02-11' then 'Montreal'
                                 when date(timestamp_date) < '2026-02-15' then 'Quebec'
                                 when date(timestamp_date) < '2026-02-18' then 'Boston'
                                 else 'Travelling' end as city

                        from activity
                        where timestamp_date between '2026-01-19' and '2026-02-20'
                        group by date(timestamp_date)
                        order by date(timestamp_date)
                        ''')

display(na_activity)

# export the following for dashboard

In [0]:
# can't load to SQL because not supported in this platform, manually download and upload, surely there is a better way?

# na_activity.createOrReplaceGlobalTempView("na_activity")